# SMS Spam Modeling Pipeline

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = SparkSession.builder.appName('sms-spam-modeling').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

clean_df = spark.read.parquet('/content/artifacts/clean_sms.parquet')
print('Rows:', clean_df.count())
print('Columns:', clean_df.columns)
clean_df.show(5, truncate=False)

Rows: 5572
Columns: ['label', 'label_text', 'message', 'msg_len_chars', 'msg_len_words']
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|label|label_text|message                                                                                                                                                    |msg_len_chars|msg_len_words|
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|0    |ham       |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                            |111          |20           |
|0    |ham       |Ok lar... Joking wif u oni...                                                    

In [3]:
train_df, test_df = clean_df.select('label', 'message').randomSplit([0.8, 0.2], seed=42)
print('Train rows:', train_df.count())
print('Test rows:', test_df.count())

tokenizer = Tokenizer(inputCol='message', outputCol='tokens')
stopwords = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_features', numFeatures=2**16)
idf = IDF(inputCol='raw_features', outputCol='features')
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=30)

pipeline = Pipeline(stages=[tokenizer, stopwords, tf, idf, lr])
print('Pipeline ready')

Train rows: 4501
Test rows: 1071
Pipeline ready


In [4]:
model = pipeline.fit(train_df)
pred_df = model.transform(test_df)

pred_df.select('label', 'prediction', 'probability', 'message').show(10, truncate=False)

+-----+----------+-------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                |message                                                                                                                                                                                                                                                                                                                                                                                         |
+-----+----------+-------------------------------------------+------------------------------------------

In [5]:
tp = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 1)).count()
tn = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 0)).count()
fp = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 1)).count()
fn = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 0)).count()

total = tp + tn + fp + fn
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')

TP=130, TN=914, FP=9, FN=18
Accuracy : 0.9748
Precision: 0.9353
Recall   : 0.8784
F1-score : 0.9059


In [6]:
from pyspark.ml import PipelineModel

model_path = '/content/artifacts/models/sms_spam_pipeline'
model.write().overwrite().save(model_path)
print('Saved model to:', model_path)

loaded_model = PipelineModel.load(model_path)

new_messages = spark.createDataFrame(
    [
        ('Congratulations! You won a free ticket. Reply WIN now!',),
        ('Can we meet tomorrow at 10 to finish the report?',),
        ('URGENT: claim your reward by calling this number now',),
    ],
    ['message']
 )

new_pred = loaded_model.transform(new_messages)
new_pred.select('message', 'prediction', 'probability').show(truncate=False)

Saved model to: /content/artifacts/models/sms_spam_pipeline
+------------------------------------------------------+----------+------------------------------------------+
|message                                               |prediction|probability                               |
+------------------------------------------------------+----------+------------------------------------------+
|Congratulations! You won a free ticket. Reply WIN now!|1.0       |[1.123837476691011E-6,0.9999988761625233] |
|Can we meet tomorrow at 10 to finish the report?      |0.0       |[0.9999999992745252,7.254747913520987E-10]|
|URGENT: claim your reward by calling this number now  |0.0       |[0.9892299285182212,0.010770071481778776] |
+------------------------------------------------------+----------+------------------------------------------+



In [7]:
errors = pred_df.filter(F.col('label') != F.col('prediction'))
print('Misclassified rows:', errors.count())
errors.select('label', 'prediction', 'probability', 'message').show(20, truncate=False)

Misclassified rows: 27
+-----+----------+-------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                |message                                                                                                                                                                                                                                                                                                                           |
+-----+----------+-------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
import pandas as pd
from pathlib import Path

metrics_path = Path('/content/artifacts/model_metrics.csv')
metrics_table = pd.DataFrame(
    [
        {'metric': 'accuracy', 'value': accuracy},
        {'metric': 'precision', 'value': precision},
        {'metric': 'recall', 'value': recall},
        {'metric': 'f1_score', 'value': f1},
    ]
)
metrics_table.to_csv(metrics_path, index=False)
print('Saved metrics table to:', metrics_path)
metrics_table

Saved metrics table to: /content/artifacts/model_metrics.csv


,metric,value
0,accuracy,0.974790
1,precision,0.935252
2,recall,0.878378
3,f1_score,0.905923


In [9]:
base_df = clean_df.select('label', 'message')
train_df_v, val_df_v, test_df_v = base_df.randomSplit([0.7, 0.15, 0.15], seed=42)

print('Train rows:', train_df_v.count())
print('Validation rows:', val_df_v.count())
print('Test rows:', test_df_v.count())

def compute_binary_metrics(predictions):
    tp = predictions.filter((F.col('label') == 1) & (F.col('prediction') == 1)).count()
    tn = predictions.filter((F.col('label') == 0) & (F.col('prediction') == 0)).count()
    fp = predictions.filter((F.col('label') == 0) & (F.col('prediction') == 1)).count()
    fn = predictions.filter((F.col('label') == 1) & (F.col('prediction') == 0)).count()

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }

def build_pipeline(classifier):
    tokenizer = Tokenizer(inputCol='message', outputCol='tokens')
    stopwords = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
    tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_features', numFeatures=2**16)
    idf = IDF(inputCol='raw_features', outputCol='features')
    return Pipeline(stages=[tokenizer, stopwords, tf, idf, classifier])

Train rows: 3979
Validation rows: 787
Test rows: 806


In [10]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.classification import LinearSVC, NaiveBayes

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
 )

lr_base = LogisticRegression(featuresCol='features', labelCol='label')
lr_pipeline = build_pipeline(lr_base)

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr_base.maxIter, [20, 30, 40, 60])
    .addGrid(lr_base.regParam, [0.0, 0.05, 0.1])
    .addGrid(lr_base.elasticNetParam, [0.0, 0.5])
    .build()
 )

lr_tvs = TrainValidationSplit(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=f1_evaluator,
    trainRatio=0.8,
    seed=42,
    parallelism=2
 )

lr_tvs_model = lr_tvs.fit(train_df_v)
best_lr = lr_tvs_model.bestModel

lr_best_idx = int(max(range(len(lr_tvs_model.validationMetrics)), key=lambda i: lr_tvs_model.validationMetrics[i]))
lr_best_map = lr_param_grid[lr_best_idx]

best_lr_params = {
    'maxIter': int(lr_best_map[lr_base.maxIter]),
    'regParam': float(lr_best_map[lr_base.regParam]),
    'elasticNetParam': float(lr_best_map[lr_base.elasticNetParam]),
 }
best_val_f1 = float(lr_tvs_model.validationMetrics[lr_best_idx])

lr_tuning_table = pd.DataFrame(
    [
        {
            'maxIter': int(param_map[lr_base.maxIter]),
            'regParam': float(param_map[lr_base.regParam]),
            'elasticNetParam': float(param_map[lr_base.elasticNetParam]),
            'val_f1': float(score),
        }
        for param_map, score in zip(lr_param_grid, lr_tvs_model.validationMetrics)
    ]
).sort_values('val_f1', ascending=False).reset_index(drop=True)

lr_tuning_table

,maxIter,regParam,elasticNetParam,val_f1
0,30,0.00,0.0,0.979782
1,30,0.00,0.5,0.979782
2,40,0.00,0.0,0.979782
3,40,0.00,0.5,0.979782
4,60,0.00,0.5,0.979782
5,60,0.00,0.0,0.979782
6,20,0.00,0.0,0.978387
7,20,0.00,0.5,0.978387
8,30,0.05,0.0,0.963833
9,60,0.05,0.0,0.963833


In [11]:
best_lr_test_pred = best_lr.transform(test_df_v)
best_lr_test_metrics = compute_binary_metrics(best_lr_test_pred)

print('Best LogisticRegression params:', best_lr_params)
print(f"Best LogisticRegression validation F1: {best_val_f1:.4f}")
best_lr_test_metrics

Best LogisticRegression params: {'maxIter': 30, 'regParam': 0.0, 'elasticNetParam': 0.0}
Best LogisticRegression validation F1: 0.9798


{'accuracy': 0.9714640198511166,
 'precision': 0.9009009009009009,
 'recall': 0.8928571428571429,
 'f1': 0.8968609865470852,
 'tp': 100,
 'tn': 683,
 'fp': 11,
 'fn': 12}

In [12]:
lsvc_base = LinearSVC(featuresCol='features', labelCol='label')
lsvc_pipeline = build_pipeline(lsvc_base)

lsvc_param_grid = (
    ParamGridBuilder()
    .addGrid(lsvc_base.maxIter, [20, 40, 60])
    .addGrid(lsvc_base.regParam, [0.01, 0.1])
    .build()
 )

lsvc_tvs = TrainValidationSplit(
    estimator=lsvc_pipeline,
    estimatorParamMaps=lsvc_param_grid,
    evaluator=f1_evaluator,
    trainRatio=0.8,
    seed=42,
    parallelism=1
 )

lsvc_tvs_model = lsvc_tvs.fit(train_df_v)
best_lsvc = lsvc_tvs_model.bestModel

lsvc_best_idx = int(max(range(len(lsvc_tvs_model.validationMetrics)), key=lambda i: lsvc_tvs_model.validationMetrics[i]))
lsvc_best_map = lsvc_param_grid[lsvc_best_idx]

best_lsvc_params = {
    'maxIter': int(lsvc_best_map[lsvc_base.maxIter]),
    'regParam': float(lsvc_best_map[lsvc_base.regParam]),
 }
best_lsvc_val_f1 = float(lsvc_tvs_model.validationMetrics[lsvc_best_idx])

lsvc_tuning_table = pd.DataFrame(
    [
        {
            'maxIter': int(param_map[lsvc_base.maxIter]),
            'regParam': float(param_map[lsvc_base.regParam]),
            'val_f1': float(score),
        }
        for param_map, score in zip(lsvc_param_grid, lsvc_tvs_model.validationMetrics)
    ]
).sort_values('val_f1', ascending=False).reset_index(drop=True)

lsvc_tuning_table

,maxIter,regParam,val_f1
0,60,0.01,0.983718
1,40,0.01,0.983718
2,60,0.10,0.983644
3,20,0.01,0.982400
4,20,0.10,0.982321
5,40,0.10,0.981004


In [13]:
nb_base = NaiveBayes(featuresCol='features', labelCol='label', modelType='multinomial')
nb_pipeline = build_pipeline(nb_base)

nb_param_grid = (
    ParamGridBuilder()
    .addGrid(nb_base.smoothing, [0.3, 0.7, 1.0, 1.5, 2.0])
    .build()
 )

nb_tvs = TrainValidationSplit(
    estimator=nb_pipeline,
    estimatorParamMaps=nb_param_grid,
    evaluator=f1_evaluator,
    trainRatio=0.8,
    seed=42,
    parallelism=1
 )

nb_tvs_model = nb_tvs.fit(train_df_v)
best_nb = nb_tvs_model.bestModel

nb_best_idx = int(max(range(len(nb_tvs_model.validationMetrics)), key=lambda i: nb_tvs_model.validationMetrics[i]))
nb_best_map = nb_param_grid[nb_best_idx]

best_nb_params = {
    'smoothing': float(nb_best_map[nb_base.smoothing]),
    'modelType': 'multinomial',
 }
best_nb_val_f1 = float(nb_tvs_model.validationMetrics[nb_best_idx])

nb_tuning_table = pd.DataFrame(
    [
        {
            'smoothing': float(param_map[nb_base.smoothing]),
            'val_f1': float(score),
        }
        for param_map, score in zip(nb_param_grid, nb_tvs_model.validationMetrics)
    ]
).sort_values('val_f1', ascending=False).reset_index(drop=True)

nb_tuning_table

,smoothing,val_f1
0,2.0,0.967244
1,1.5,0.966123
2,1.0,0.958745
3,0.7,0.953899
4,0.3,0.942009


In [14]:
best_lsvc_test_pred = best_lsvc.transform(test_df_v)
best_lsvc_test_metrics = compute_binary_metrics(best_lsvc_test_pred)

best_nb_test_pred = best_nb.transform(test_df_v)
best_nb_test_metrics = compute_binary_metrics(best_nb_test_pred)

comparison_rows = [
    {'model': 'LogisticRegression_tuned', 'best_params': best_lr_params, 'val_f1': best_val_f1, **best_lr_test_metrics},
    {'model': 'LinearSVC_tuned', 'best_params': best_lsvc_params, 'val_f1': best_lsvc_val_f1, **best_lsvc_test_metrics},
    {'model': 'NaiveBayes_tuned', 'best_params': best_nb_params, 'val_f1': best_nb_val_f1, **best_nb_test_metrics},
]

comparison_table = pd.DataFrame(comparison_rows).sort_values('f1', ascending=False).reset_index(drop=True)

model_candidates = [
    ('LogisticRegression_tuned', best_lr, best_lr_test_metrics),
    ('LinearSVC_tuned', best_lsvc, best_lsvc_test_metrics),
    ('NaiveBayes_tuned', best_nb, best_nb_test_metrics),
]
best_model_name, best_model, best_model_metrics = max(model_candidates, key=lambda x: x[2]['f1'])

comparison_path = Path('/content/artifacts/model_comparison.csv')
comparison_table.to_csv(comparison_path, index=False)

best_model_path = '/content/artifacts/models/sms_spam_best_pipeline'
best_model.write().overwrite().save(best_model_path)

best_model_summary = pd.DataFrame([{'model': best_model_name, **best_model_metrics}])
best_model_summary_path = Path('/content/artifacts/best_model_metrics.csv')
best_model_summary.to_csv(best_model_summary_path, index=False)

print('Saved model comparison to:', comparison_path)
print('Saved best model to:', best_model_path)
print('Saved best model metrics to:', best_model_summary_path)
comparison_table

Saved model comparison to: /content/artifacts/model_comparison.csv
Saved best model to: /content/artifacts/models/sms_spam_best_pipeline
Saved best model metrics to: /content/artifacts/best_model_metrics.csv


,model,best_params,val_f1,accuracy,precision,recall,f1,tp,tn,fp,fn
0,LinearSVC_tuned,"{'maxIter': 40, 'regParam': 0.01}",0.983718,0.977667,0.935185,0.901786,0.918182,101,687,7,11
1,LogisticRegression_tuned,"{'maxIter': 30, 'regParam': 0.0, 'elasticNetPa...",0.979782,0.971464,0.900901,0.892857,0.896861,100,683,11,12
2,NaiveBayes_tuned,"{'smoothing': 2.0, 'modelType': 'multinomial'}",0.967244,0.965261,0.833333,0.937500,0.882353,105,673,21,7


In [15]:
print(comparison_table[['model', 'accuracy', 'precision', 'recall', 'f1']].to_string(index=False))
print('Best model:', best_model_name)
print('Best model metrics:', best_model_metrics)

                   model  accuracy  precision   recall       f1
         LinearSVC_tuned  0.977667   0.935185 0.901786 0.918182
LogisticRegression_tuned  0.971464   0.900901 0.892857 0.896861
        NaiveBayes_tuned  0.965261   0.833333 0.937500 0.882353
Best model: LinearSVC_tuned
Best model metrics: {'accuracy': 0.9776674937965261, 'precision': 0.9351851851851852, 'recall': 0.9017857142857143, 'f1': 0.9181818181818182, 'tp': 101, 'tn': 687, 'fp': 7, 'fn': 11}


### Download the **models** directory from colab.

In [17]:
import shutil
from google.colab import files

models_path = '/content/artifacts/models'

# Zip the folder
shutil.make_archive("models", "zip", models_path)
# Download the zip
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>